# 14 October 2026 - X. schlechteri revised gene models

The new X. schlechteri seedling counts table (`all_data/to_add/X schlechteri/`) uses revised gene models that weren't in the DB. The biologist sent their sequences in `all_data/to_add/Xsch_47_updated_seqs.fasta` (47 records):
- **40** `_A_NEW` / `_B_NEW` / `_C_NEW` / `_NEW` genes that are in the counts table but not the DB
- **6** more split parts that aren't in the counts/DEG/GRN files (they get a sequence but no expression data)
- **1** revised sequence for an existing gene, `Xsch.v2.MJHO01000004.1.247` (same name, so only `coding_sequence` changes)

These replace **22** old gene models (e.g. `Xsch.v2.MJHO01000001.1.830` -> `..._A_NEW` + `..._B_NEW`).

**Run this notebook before the "14 October 2026" loading cells in `workflow_notes.ipynb`**, so the 40 genes' expression and DEG rows are loaded in the same pass.

Steps (plan: `docs/XS_DATA_PLAN.md` Step 3):
1. Add / update the 47 sequences - **done here**
2. Add annotations - **waiting for a blast2go export from the biologist**
3. Delete the 22 old gene models - **waiting for confirmation from the biologist**

In [1]:
import os
import re
from pathlib import Path

import pandas as pd
from Bio import SeqIO

# db.DB() uses a relative sqlite path. Run from the repo root, otherwise a new empty DB is silently created
if Path.cwd().name == "notebooks":
    os.chdir("..")
assert Path("database/data/all_xerophyta_species_db.sqlite").exists(), f"Run from the repo root (cwd is {Path.cwd()})"

import database.db as db
import database.db_manager as dbm

SPECIES_NAME = "X. schlechteri"
FASTA_FILE = Path("all_data/to_add/Xsch_47_updated_seqs.fasta")
COUNTS_FILE = Path("all_data/to_add/X schlechteri/Xs_seedlings_v4c_16_07_26 DESeq2 normalised counts table.csv")

database = db.DB()
species_id = database.get_species_by_name(SPECIES_NAME).id

### Check the FASTA against the DB and the counts table

In [2]:
records = {r.id: str(r.seq) for r in SeqIO.parse(FASTA_FILE, "fasta")}
db_genes = dbm.map_genes_to_ids(species_id)
counts_genes = set(pd.read_csv(COUNTS_FILE, usecols=[0]).iloc[:, 0])

new_genes = sorted(set(records) - set(db_genes))
updated_genes = sorted(set(records) & set(db_genes))
missing_from_db = sorted(counts_genes - set(db_genes))

print(f"{len(records)} sequences: {len(new_genes)} new genes, {len(updated_genes)} existing genes to update {updated_genes}")
print(f"Counts-table genes missing from the DB: {len(missing_from_db)}, covered by the FASTA: {len(set(missing_from_db) & set(records))}")
print(f"New genes not in the counts table (no expression data): {sorted(set(new_genes) - counts_genes)}")
assert set(missing_from_db) <= set(records), "some counts-table genes still have no sequence"

# old gene models replaced by the revised ones (strip the _A_NEW / _B_NEW / _C_NEW / _NEW suffix)
old_genes = sorted({re.sub(r"_([A-C]_)?NEW$", "", g) for g in missing_from_db})
print(f"{len(old_genes)} old gene models replaced: {old_genes}")

47 sequences: 46 new genes, 1 existing genes to update ['Xsch.v2.MJHO01000004.1.247']
Counts-table genes missing from the DB: 40, covered by the FASTA: 40
New genes not in the counts table (no expression data): ['Xsch.v2.MJHO01000001.1.274_A_NEW', 'Xsch.v2.MJHO01000010.1.471_B_NEW', 'Xsch.v2.MJHO01000043.1.103_A_NEW', 'Xsch.v2.MJHO01000043.1.103_C_NEW', 'Xsch.v2.MJHO01000054.1.132_A_NEW', 'Xsch.v2.MJHO01000060.1.126_B_NEW']
22 old gene models replaced: ['Xsch.v2.MJHO01000001.1.274', 'Xsch.v2.MJHO01000001.1.830', 'Xsch.v2.MJHO01000002.1.375', 'Xsch.v2.MJHO01000002.1.491', 'Xsch.v2.MJHO01000002.1.684', 'Xsch.v2.MJHO01000002.1.764', 'Xsch.v2.MJHO01000004.1.141', 'Xsch.v2.MJHO01000004.1.296', 'Xsch.v2.MJHO01000010.1.471', 'Xsch.v2.MJHO01000012.1.311', 'Xsch.v2.MJHO01000013.1.141', 'Xsch.v2.MJHO01000013.1.77', 'Xsch.v2.MJHO01000043.1.103', 'Xsch.v2.MJHO01000054.1.132', 'Xsch.v2.MJHO01000055.1.111', 'Xsch.v2.MJHO01000060.1.126', 'Xsch.v2.MJHO01000060.1.13', 'Xsch.v2.MJHO01000060.1.288', 'Xsc

### 1. Add / update the sequences
`add_gene_sequence_from_fasta` upserts on `gene_name`, so the new genes are added and `Xsch.v2.MJHO01000004.1.247` gets its revised sequence.

In [3]:
dbm.add_gene_sequence_from_fasta(str(FASTA_FILE), species_id)

db_genes = dbm.map_genes_to_ids(species_id)
print(f"Counts-table genes still missing from the DB: {len(counts_genes - set(db_genes))}")
gene = db.DB().get_gene_by_name("Xsch.v2.MJHO01000004.1.247")
assert gene.coding_sequence == records[gene.gene_name], "revised sequence not updated"

Adding gene sequences for X. schlechteri from all_data/to_add/Xsch_47_updated_seqs.fasta
Processed 1/47 records
Processed 2/47 records
Processed 3/47 records
Processed 4/47 records
Processed 5/47 records
Processed 6/47 records
Processed 7/47 records
Processed 8/47 records
Processed 9/47 records
Processed 10/47 records
Processed 11/47 records
Processed 12/47 records
Processed 13/47 records
Processed 14/47 records
Processed 15/47 records
Processed 16/47 records
Processed 17/47 records
Processed 18/47 records
Processed 19/47 records
Processed 20/47 records
Processed 21/47 records
Processed 22/47 records
Processed 23/47 records
Processed 24/47 records
Processed 25/47 records
Processed 26/47 records
Processed 27/47 records
Processed 28/47 records
Processed 29/47 records
Processed 30/47 records
Processed 31/47 records
Processed 32/47 records
Processed 33/47 records
Processed 34/47 records
Processed 35/47 records
Processed 36/47 records
Processed 37/47 records
Processed 38/47 records
Processe

Counts-table genes still missing from the DB: 0


### 2. Add annotations - waiting on the biologist
No blast2go annotation export came with the sequences, so the new genes have no description / GO terms on the gene info page yet. When it arrives (same format as `all_data/Xschlechteri_Nov2024/20241108_Xschlechteri_annotation_23009_export_table_Oliver.csv`), check for blank `GO IDs` / `GO Names` first: `parse_annotations` doesn't handle missing values in those columns.

In [4]:
# ANNOTATION_FILE = "all_data/to_add/<blast2go export>.csv"
# dbm.add_gene_annotations(ANNOTATION_FILE, species_id, sep=",")

### 3. Delete the 22 old gene models - waiting on the biologist
Leaving them in is harmless (they just have no seedling expression data). Deleting a gene cascades to its annotations, expression, DEG and GRN rows. Always do the dry run first.

In [5]:
# summary = db.DB().delete_genes_by_names(old_genes, dry_run=True)
# summary

In [6]:
# summary = db.DB().delete_genes_by_names(old_genes, dry_run=False)
# summary